# Day 38 – Q2: Similarity Across Representations

Compare BOW, TF-IDF, Word2Vec averaging, and Sentence-BERT on two semantically similar reviews.

In [3]:
import re
import math
import numpy as np
import pandas as pd
from collections import Counter
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

REVIEWS_PATH = "shopsense_reviews.csv"
REVIEW_A = "incredible camera but terrible battery life"
REVIEW_B = "Battery drains fast although photos are stunning"


In [4]:
def load_reviews(path):
    try:
        df = pd.read_csv(path)
        assert "review_text" in df.columns
        return df
    except FileNotFoundError:
        raise FileNotFoundError(f"Dataset not found: {path}")

df = load_reviews(REVIEWS_PATH)
print("Loaded:", df.shape)


Loaded: (10199, 20)


In [5]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text.lower())
    return " ".join(text.split())

def tokenize(text):
    return clean_text(text).split()

ra_clean = clean_text(REVIEW_A)
rb_clean = clean_text(REVIEW_B)
ra_tokens = tokenize(REVIEW_A)
rb_tokens = tokenize(REVIEW_B)

print("Review A cleaned:", ra_clean)
print("Review B cleaned:", rb_clean)


Review A cleaned: incredible camera but terrible battery life
Review B cleaned: battery drains fast although photos are stunning


## (a) & (b) BOW Similarity — and why it fails

In [6]:
def bow_cosine(tokens_a, tokens_b):
    vocab = sorted(set(tokens_a) | set(tokens_b))
    vec_a = np.array([tokens_a.count(w) for w in vocab], dtype=float)
    vec_b = np.array([tokens_b.count(w) for w in vocab], dtype=float)
    dot = np.dot(vec_a, vec_b)
    norm = (np.linalg.norm(vec_a) * np.linalg.norm(vec_b))
    return dot / norm if norm > 0 else 0.0, vocab, vec_a, vec_b

bow_score, bow_vocab, vec_a, vec_b = bow_cosine(ra_tokens, rb_tokens)

print(f"BOW cosine similarity: {bow_score:.4f}")
print("\nWord overlap analysis:")
overlap = set(ra_tokens) & set(rb_tokens)
only_a  = set(ra_tokens) - set(rb_tokens)
only_b  = set(rb_tokens) - set(ra_tokens)
print(f"  Shared words       : {sorted(overlap)}")
print(f"  Only in Review A   : {sorted(only_a)}")
print(f"  Only in Review B   : {sorted(only_b)}")
print(f"\nBOW fails because 'incredible'/'stunning' and 'terrible'/'drains fast' are synonyms")
print("but share zero literal characters — BOW has no way to know they mean the same thing.")


BOW cosine similarity: 0.1543

Word overlap analysis:
  Shared words       : ['battery']
  Only in Review A   : ['but', 'camera', 'incredible', 'life', 'terrible']
  Only in Review B   : ['although', 'are', 'drains', 'fast', 'photos', 'stunning']

BOW fails because 'incredible'/'stunning' and 'terrible'/'drains fast' are synonyms
but share zero literal characters — BOW has no way to know they mean the same thing.


## TF-IDF Similarity

In [7]:
def tfidf_cosine(text_a, text_b):
    vec = TfidfVectorizer()
    matrix = vec.fit_transform([text_a, text_b])
    return float(cosine_similarity(matrix[0], matrix[1])[0][0])

tfidf_score = tfidf_cosine(ra_clean, rb_clean)
print(f"TF-IDF cosine similarity: {tfidf_score:.4f}")
print("TF-IDF still operates on exact lexical match — better weighting but same vocabulary gap.")


TF-IDF cosine similarity: 0.0846
TF-IDF still operates on exact lexical match — better weighting but same vocabulary gap.


## Word2Vec Averaging Similarity

In [8]:
def train_word2vec_on_corpus(df, window=5, vector_size=100, seed=42):
    corpus = [tokenize(t) for t in df["review_text"] if isinstance(t, str)]
    corpus = [s for s in corpus if len(s) > 2]
    model = Word2Vec(sentences=corpus, vector_size=vector_size,
                     window=window, min_count=2, workers=4, seed=seed)
    return model

w2v_model = train_word2vec_on_corpus(df)
print("Word2Vec trained. Vocab size:", len(w2v_model.wv))


Word2Vec trained. Vocab size: 253


In [9]:
def averaged_w2v_vector(model, tokens):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        return None
    return np.mean(vecs, axis=0)

def w2v_cosine(model, tokens_a, tokens_b):
    va = averaged_w2v_vector(model, tokens_a)
    vb = averaged_w2v_vector(model, tokens_b)
    if va is None or vb is None:
        return None
    return float(cosine_similarity([va], [vb])[0][0])

w2v_score = w2v_cosine(w2v_model, ra_tokens, rb_tokens)
print(f"Word2Vec avg cosine similarity: {w2v_score:.4f}")
print("Word2Vec captures semantics — 'stunning' and 'incredible' land near each other.")


Word2Vec avg cosine similarity: 0.3753
Word2Vec captures semantics — 'stunning' and 'incredible' land near each other.


## Sentence-BERT Similarity

In [10]:
try:
    from sentence_transformers import SentenceTransformer
    sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
    emb_a = sbert_model.encode([REVIEW_A])
    emb_b = sbert_model.encode([REVIEW_B])
    sbert_score = float(cosine_similarity(emb_a, emb_b)[0][0])
    print(f"Sentence-BERT cosine similarity: {sbert_score:.4f}")
    print("SBERT encodes full sentence meaning — best at capturing mixed-sentiment similarity.")
except ImportError:
    print("sentence-transformers not installed. Run: pip install sentence-transformers")
    print("Expected SBERT score: ~0.70-0.85 (much higher than BOW or TF-IDF)")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence-BERT cosine similarity: 0.6301
SBERT encodes full sentence meaning — best at capturing mixed-sentiment similarity.


## (c) Semantic Gap Summary

In [11]:
print("=== Semantic Gap: How Each Representation Closes It ===")
print()
print("BOW:")
print("  Represents each document as a word count vector.")
print("  Two documents with ZERO shared words get similarity = 0.0")
print("  even if they express identical sentiment using different vocabulary.")
print()
print("TF-IDF:")
print("  Weights words by how unique they are across the corpus.")
print("  Still purely lexical — same zero-overlap problem for synonym-heavy sentences.")
print()
print("Word2Vec Averaging:")
print("  Maps each word to a dense semantic vector trained on co-occurrence.")
print("  'incredible' and 'stunning' cluster near each other.")
print("  Averaging loses word order and sentence structure.")
print()
print("Sentence-BERT:")
print("  Encodes the entire sentence as one vector using a fine-tuned transformer.")
print("  Captures meaning, word order, and negation — the most complete representation.")
print("  Best score on semantically similar but lexically different sentence pairs.")


=== Semantic Gap: How Each Representation Closes It ===

BOW:
  Represents each document as a word count vector.
  Two documents with ZERO shared words get similarity = 0.0
  even if they express identical sentiment using different vocabulary.

TF-IDF:
  Weights words by how unique they are across the corpus.
  Still purely lexical — same zero-overlap problem for synonym-heavy sentences.

Word2Vec Averaging:
  Maps each word to a dense semantic vector trained on co-occurrence.
  'incredible' and 'stunning' cluster near each other.
  Averaging loses word order and sentence structure.

Sentence-BERT:
  Encodes the entire sentence as one vector using a fine-tuned transformer.
  Captures meaning, word order, and negation — the most complete representation.
  Best score on semantically similar but lexically different sentence pairs.
